In [1]:
from configparser import ConfigParser
import os 
import uuid

In [2]:
# NOTE: actual function defined in ../utils/general.py
def get_mac_address() -> str:
    """Returns the device's MAC address in the format "AB:CD:EF:GH:00"."""
    mac = uuid.getnode()
    return ':'.join(f'{(mac >> i) & 0xff:02x}' for i in range(0, 48, 8))


**NOTE:** simulating the "/ui/signup" endpoint.

In [3]:
identity_config:ConfigParser = ConfigParser()
identity_config.read('../config/identity.conf')

# Check if there is already a common name for this user (i.e. they already have an account)
if identity_config['IDENTITY']['COMMON_NAME']: 
    raise Exception('ACCOUNT EXISTS')           # abort(409)
    
# Extract the required keys
# NOTE: extracted from req in actual function
new_common_name:str = 'jjhealey'
new_allocated_storage:int = '50'
new_peer_storage_path:str = 'tmp-peer-storage/'

# Check that the required keys were given, and return bad request if wrong
try:
    
    # Check that keys are given 
    if not (new_common_name and new_allocated_storage and new_peer_storage_path): raise AttributeError
    
    # Make sure allocated_storage is an integer
    new_allocated_storage = int(new_allocated_storage)

except: 
    # Bad request (missing/invalid info) 
    raise Exception('BAD REQUEST')      # abort(400)

# --- Updating identity --- #
# Update the identity config with the new common name, mac, allocated storage, and peer storage path
identity_config['IDENTITY']['COMMON_NAME'] = new_common_name
identity_config['IDENTITY']['MAC'] = get_mac_address() 
identity_config['SETTINGS']['ALLOCATED_STORAGE'] = str(new_allocated_storage)
identity_config['PATHS']['PEER_STORAGE_PATH'] = new_peer_storage_path    

# --- Saving new info --- #
# Create the [new_peer_storage_path] if it does not exist
os.makedirs(new_peer_storage_path, exist_ok=True)

# Save the updated identity dict
with open('../config/identity.conf', 'w') as file: 
    identity_config.write(file)

# Return
print('\033[92mSUCCESS.\033[0m')    # return jsonify(...)

Exception: ACCOUNT EXISTS